02 v2 — OOF - Modelo de incrustación (embedding) de imágenes con CLIP (DEFINITIVO).

Cambios al  codigo de Ani:
- Fix PCA dentro del fold: el PCA ahora se fittea solo con datos de train de cada fold, eliminando leakage de validación

- Early stopping LightGBM: el modelo corta solo cuando la métrica no mejora en 50 rondas en lugar de correr siempre 500 árboles fijos

- Múltiples imágenes por propiedad: en vez de usar solo la primera foto, se promedian los embeddings CLIP de todas las fotos disponibles de cada propiedad

- Bug fix preprocess CLIP: se separó el pipeline de transforms para hacerlo compatible con augmentation futura, usando la normalización correcta de CLIP

- CLIP text embeddings: se generaron embeddings de 24 frases descriptivas de propiedades (calidad, estado, estilo, amenities Florida-specific) y se calculó la similitud coseno con cada imagen, agregando esas similitudes como features adicionales al LightGBM

- Concatenación visual + semántica: los 512 dims de embedding visual se concatenaron con las 24 similitudes de texto antes del PCA, dando un vector de 536 dims por propiedad

In [ ]:
# Instalar open_clip
!pip install -q open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00


In [ ]:
import os
import zipfile

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold

import torch
import open_clip
from torch.utils.data import Dataset, DataLoader
from PIL import Image

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ZIP_PATH = "/content/drive/MyDrive/participant/participant.zip"
SUBMISSIONS_DRIVE_DIR = "/content/drive/MyDrive/participant/submissions"

Mounted at /content/drive


In [ ]:
# Identificar y extraer rutas reales dentro del ZIP
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    all_files = zip_ref.namelist()

    print("Extrayendo metadatos y fotos al disco local (esto puede tardar)...")
    for file in all_files:
        if file.endswith('.csv') or "data/images/" in file:
            zip_ref.extract(file, path="/content/")
    print("Extracción completada en /content/")

def find_path(suffix):
    for f in all_files:
        if f.endswith(suffix):
            return os.path.join("/content", f)
    return None

path_train_meta = find_path("train_photo_metadata.csv")
path_test_meta = find_path("test_photo_metadata.csv")
path_train_tab = find_path("train_processed.csv")
path_test_tab = find_path("test_processed.csv")

print(f"\ntrain_meta: {path_train_meta}")
print(f"test_meta:  {path_test_meta}")
print(f"train_tab:  {path_train_tab}")
print(f"test_tab:   {path_test_tab}")

Extrayendo metadatos y fotos al disco local (esto puede tardar)...
Extracción completada en /content/

train_meta: /content/participant/data/train_photo_metadata.csv
test_meta:  /content/participant/data/test_photo_metadata.csv
train_tab:  /content/participant/data/tabular/train_processed.csv
test_tab:   /content/participant/data/tabular/test_processed.csv


In [ ]:
if os.path.exists(ZIP_PATH):
    print("Archivo encontrado")
else:
    print("El archivo NO está en esa ruta.")
    # Listar contenido para ver qué hay realmente
    if os.path.exists("/content/drive/MyDrive/participant"):
        print("Contenido de la carpeta participant:", os.listdir("/content/drive/MyDrive/participant"))
    else:
        print("Ni siquiera existe la carpeta 'participant' en MyDrive")

✅ ¡Archivo encontrado!


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("Estás corriendo sin GPU. CLIP va a tardar mucho.")
    print("   Activá GPU en: Runtime → Change runtime type → T4 GPU")

Device: cuda


In [ ]:
# Load metadata y arreglar paths
def fix_img_path(p):
    if p.startswith("/content/participant/"):
        return p
    if p.startswith("data/"):
        return os.path.join("/content/participant", p)
    if p.startswith("/content/data/"):
        return p.replace("/content/data/", "/content/participant/data/")
    return p

train_meta = pd.read_csv(path_train_meta)
test_meta  = pd.read_csv(path_test_meta)

train_meta['image_path'] = train_meta['image_path'].apply(fix_img_path)
test_meta['image_path']  = test_meta['image_path'].apply(fix_img_path)

train_tab = pd.read_csv(path_train_tab)
test_tab  = pd.read_csv(path_test_tab)

TARGET = "log_price"

# Todas las imágenes (múltiples filas por zpid)
train_img_all = train_tab[["zpid", TARGET]].merge(
    train_meta[["zpid", "image_path"]], on="zpid", how="inner")
test_img_all = test_tab[["zpid"]].merge(
    test_meta[["zpid", "image_path"]], on="zpid", how="inner")

# Una fila por propiedad
train_img = train_tab[["zpid", TARGET]].merge(
    train_meta[["zpid"]].drop_duplicates(), on="zpid", how="inner")
test_img = test_tab[["zpid"]].merge(
    test_meta[["zpid"]].drop_duplicates(), on="zpid", how="inner")

# Cargar descripciones textuales por zpid
# "description" es la columna del listado de Zillow — señal semántica personalizada
train_desc = train_tab[["zpid", "description"]].copy()
test_desc  = test_tab[["zpid", "description"]].copy()

# Fallback: si description es NaN, usar string vacío
train_desc["description"] = train_desc["description"].fillna("").astype(str)
test_desc["description"]  = test_desc["description"].fillna("").astype(str)

# Verificación
print("Verificación de paths:")
for p in train_img_all["image_path"].head(3).tolist():
    print(f"  {os.path.exists(p)} | {p}")

if not os.path.exists(train_img_all["image_path"].iloc[0]):
    raise FileNotFoundError(
        "Los paths de imágenes no existen. Verificá que el ZIP "
        "se haya extraído correctamente en /content/participant/"
    )

print(f"\nTrain: {len(train_img)} propiedades, {len(train_img_all)} imágenes en total")
print(f"Test:  {len(test_img)} propiedades, {len(test_img_all)} imágenes en total")
print(f"Descripciones train no vacías: {(train_desc['description'] != '').sum()} / {len(train_desc)}")

Verificación de paths:
  True | /content/participant/data/images/props/1011541_007.jpg
  True | /content/participant/data/images/props/1005445_007.jpg
  True | /content/participant/data/images/props/1002822_007.jpg

Train: 11834 propiedades, 134921 imágenes en total
Test:  5035 propiedades, 56843 imágenes en total
Descripciones train no vacías: 11840 / 11840


In [ ]:
# Cargar CLIP
model_clip, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='openai',
)
model_clip = model_clip.to(DEVICE).eval()
print(f"CLIP cargado. Embedding dim: 512")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP cargado. Embedding dim: 512


In [ ]:
# Image dataset
# Separamos el preprocess de CLIP en dos partes:
# - spatial: resize/crop (lo reemplazamos con versión augmentable)
# - normalize: la normalización final de CLIP (se mantiene intacta)
import torchvision.transforms as T

clip_normalize = T.Normalize(
    mean=(0.48145466, 0.4578275, 0.40821073),
    std=(0.26862954, 0.26130258, 0.27577711)
)

transform_base = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    clip_normalize,
])

transform_aug = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.RandomRotation(degrees=10),
    T.CenterCrop(224),
    T.ToTensor(),
    clip_normalize,
])

class PropertyImageDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.paths = image_paths
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.transform(img)
        except Exception:
            return torch.zeros(3, 224, 224)

In [ ]:
# Extract embeddings con CLIP
def extract_embeddings(image_paths, batch_size=128):
    #ds = PropertyImageDataset(image_paths, preprocess)
    ds = PropertyImageDataset(image_paths, transform_base)
    dl = DataLoader(ds, batch_size=batch_size, num_workers=4, pin_memory=True)
    all_embs = []
    with torch.no_grad():
        for batch in dl:
            embs = model_clip.encode_image(batch.to(DEVICE))
            embs = embs / embs.norm(dim=-1, keepdim=True)
            all_embs.append(embs.cpu().float().numpy())
    return np.vstack(all_embs)

def average_embeddings_by_zpid(img_df, embs):
    """Promedia los embeddings de todas las imágenes de cada propiedad."""
    img_df = img_df.copy().reset_index(drop=True)
    img_df["emb_idx"] = range(len(img_df))
    result = []
    for zpid, group in img_df.groupby("zpid", sort=False):
        idxs = group["emb_idx"].values
        result.append({
            "zpid": zpid,
            "embedding": embs[idxs].mean(axis=0)  # promedio de todas las fotos
        })
    return pd.DataFrame(result)

print("Extracting train embeddings...")
train_embs_all = extract_embeddings(train_img_all["image_path"].tolist())

# Sanity check
if np.allclose(train_embs_all[0], train_embs_all[1]):
    raise RuntimeError(
        "Los embeddings son todos iguales. "
        "Indica que las imágenes se cargaron como tensores de ceros (paths rotos)."
    )

print("Extracting test embeddings...")
test_embs_all = extract_embeddings(test_img_all["image_path"].tolist())

# Promediar por propiedad
print("Averaging embeddings by property...")
train_emb_df = average_embeddings_by_zpid(train_img_all[["zpid", "image_path"]], train_embs_all)
test_emb_df  = average_embeddings_by_zpid(test_img_all[["zpid", "image_path"]], test_embs_all)

# Alinear con train_img / test_img (orden correcto para el OOF)
train_embs = np.vstack(
    train_emb_df.set_index("zpid").loc[train_img["zpid"].values, "embedding"].values)
test_embs = np.vstack(
    test_emb_df.set_index("zpid").loc[test_img["zpid"].values, "embedding"].values)

print(f"train_embs shape: {train_embs.shape}")
print(f"test_embs shape:  {test_embs.shape}")

Extracting train embeddings...
Extracting test embeddings...
Averaging embeddings by property...
train_embs shape: (11834, 512)
test_embs shape:  (5035, 512)


In [ ]:
# CLIP Text Embeddings — descripciones reales por propiedad
# En vez de frases fijas, usamos la descripción real del listado de Zillow.
# Esto genera un embedding de texto PERSONALIZADO por propiedad que CLIP
# puede comparar directamente con sus imágenes en el mismo espacio vectorial.

tokenizer = open_clip.get_tokenizer('ViT-B-32')

def encode_descriptions(descriptions, batch_size=256):
    """Encodea descripciones de texto con CLIP en batches."""
    all_embs = []
    for i in range(0, len(descriptions), batch_size):
        batch = descriptions[i:i+batch_size]
        # CLIP tokenizer trunca a 77 tokens automáticamente
        tokens = tokenizer(batch).to(DEVICE)
        with torch.no_grad():
            embs = model_clip.encode_text(tokens)
            embs = embs / embs.norm(dim=-1, keepdim=True)
        all_embs.append(embs.cpu().float().numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Encoded {min(i+batch_size, len(descriptions))}/{len(descriptions)}")
    return np.vstack(all_embs)

# Alinear descripciones con el orden de train_img y test_img
train_descriptions = train_desc.set_index("zpid").loc[
    train_img["zpid"].values, "description"].values.tolist()
test_descriptions = test_desc.set_index("zpid").loc[
    test_img["zpid"].values, "description"].values.tolist()

print("Encoding train descriptions...")
train_text_embs = encode_descriptions(train_descriptions)

print("Encoding test descriptions...")
test_text_embs = encode_descriptions(test_descriptions)

print(f"Train text embs shape: {train_text_embs.shape}")

# Similitud coseno imagen ↔ descripción propia (1 feature por propiedad)
# Esta es la señal más directa: ¿cuánto coincide la foto con lo que describe el vendedor?
train_img_text_sim = (train_embs * train_text_embs).sum(axis=1, keepdims=True)
test_img_text_sim  = (test_embs  * test_text_embs).sum(axis=1, keepdims=True)

# Concatenar: embedding visual + embedding de texto + similitud imagen-texto
train_embs_full = np.hstack([train_embs, train_text_embs, train_img_text_sim])
test_embs_full  = np.hstack([test_embs,  test_text_embs,  test_img_text_sim])

print(f"train_embs_full shape: {train_embs_full.shape}")  # 512 + 512 + 1 = 1025
print(f"test_embs_full shape:  {test_embs_full.shape}")

Encoding train descriptions...
  Encoded 256/11834
  Encoded 2816/11834
  Encoded 5376/11834
  Encoded 7936/11834
  Encoded 10496/11834
Encoding test descriptions...
  Encoded 256/5035
  Encoded 2816/5035
Train text embs shape: (11834, 512)
train_embs_full shape: (11834, 1025)
test_embs_full shape:  (5035, 1025)


In [ ]:
# OOF Training & Prediction (Images)
N_COMPONENTS = 128
kf = KFold(n_splits=5, shuffle=True, random_state=42)

train_oof_img = np.zeros(len(train_img))
test_preds_img = np.zeros(len(test_img))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_img)):

    # PCA dentro del fold (Bug 1 fix — sin leakage)
    pca = PCA(n_components=N_COMPONENTS, random_state=42)
    X_train_f = pca.fit_transform(train_embs_full[train_idx])
    X_val_f   = pca.transform(train_embs_full[val_idx])
    X_test_f  = pca.transform(test_embs_full)                # test con el mismo PCA del fold

    if fold == 0:
        print(f"Varianza explicada con {N_COMPONENTS} componentes: {pca.explained_variance_ratio_.sum():.3f}")

    y_train_f = train_img[TARGET].values[train_idx]
    y_val_f   = train_img[TARGET].values[val_idx]

    # Early stopping (Bug 2 fix)
    model_img = lgb.LGBMRegressor(
        n_estimators=1000,          # techo alto, early stopping corta antes
        learning_rate=0.05,
        num_leaves=63,
        random_state=42,
        verbosity=-1
    )

    model_img.fit(
        X_train_f, y_train_f,
        eval_set=[(X_val_f, y_val_f)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    best = model_img.best_iteration_
    train_oof_img[val_idx] = model_img.predict(X_val_f)
    test_preds_img += model_img.predict(X_test_f) / kf.n_splits

    print(f"Fold {fold+1} — best iteration: {best}")

Varianza explicada con 128 componentes: 0.850


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 — best iteration: 369


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 — best iteration: 382


...

Fold 3 — best iteration: 348


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 — best iteration: 498


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 — best iteration: 445


In [ ]:
# Evaluate OOF
train_price_img = train_tab.set_index("zpid").loc[train_img["zpid"].values, "lastSoldPrice_hpi_adjusted"].values
pred_price_oof_img = np.expm1(train_oof_img)

print(f"\nImage OOF R² (log): {r2_score(train_img[TARGET], train_oof_img):.4f}")
print(f"Image OOF MAE ($):  ${mean_absolute_error(train_price_img, pred_price_oof_img):,.0f}")
print(f"Image OOF MAPE (%): {np.mean(np.abs((train_price_img - pred_price_oof_img) / train_price_img)) * 100:.1f}%")


Image OOF R² (log): 0.4315
Image OOF MAE ($):  $191,739
Image OOF MAPE (%): 43.9%


In [ ]:
# Save OOF (Practice) & Test Submission
median_pred = np.median(pred_price_oof_img)

# OOF (alineado con TODO el train, fallback con mediana para zpid sin imagen)
oof_full = train_tab[["zpid"]].copy()
oof_partial = pd.DataFrame({
    "zpid": train_img["zpid"].values,
    "predicted_price": pred_price_oof_img,
})
oof_full = oof_full.merge(oof_partial, on="zpid", how="left")
oof_full["predicted_price"] = oof_full["predicted_price"].fillna(median_pred)

oof_file = os.path.join(SUBMISSIONS_DRIVE_DIR, "image_clip_oof.csv")
oof_full.to_csv(oof_file, index=False)

# Test
test_full = test_tab[["zpid"]].copy()
test_partial = pd.DataFrame({
    "zpid": test_img["zpid"].values,
    "predicted_price": np.expm1(test_preds_img),
})
test_full = test_full.merge(test_partial, on="zpid", how="left")
test_full["predicted_price"] = test_full["predicted_price"].fillna(median_pred)

test_file = os.path.join(SUBMISSIONS_DRIVE_DIR, "jaime_clip_04")
test_full.to_csv(test_file, index=False)

print(f"\nGuardado OOF: {oof_file} ({len(oof_full)} filas)")
print(f"Guardado Test: {test_file} ({len(test_full)} filas)")


Guardado OOF: /content/drive/MyDrive/participant/submissions/image_clip_oof.csv (11840 filas)
Guardado Test: /content/drive/MyDrive/participant/submissions/jaime_clip_04 (5038 filas)
